# Ordinary Kriging Stage 1 — Indicator Kriging Visualization

**What this notebook does:**
1. Loads the **global indicator variogram** (spherical, single variogram pooled across all days — Haylock 2008, §3.3)
2. For several selected days with partial rainfall (wet/dry boundaries), visualizes:
   - Full domain P(rain) map
   - Zoomed wet/dry boundary with variogram range circle, showing which stations fall inside/outside
   - Binary wet/dry classification (P > 0.4 threshold, Hofstra 2008)
3. Shows how the global indicator variogram parameters remain constant across all days, unlike stage-2 amount kriging (which varies by transform)

**Inputs (read from S3):**
- `s3://thesis-data-ismaktam/kriging/global_variogram.pkl` (created by `kriging_train.ipynb`)

**Local data:**
- ReKIS station network, all 1966 train stations (492 holdout excluded)

## 0. Imports + paths

In [1]:
import sys, os, json, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import boto3

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
})

_NB = Path.cwd()
while _NB != _NB.parent and not (_NB / 'pyproject.toml').exists():
    _NB = _NB.parent
if str(_NB / 'src') not in sys.path:
    sys.path.insert(0, str(_NB / 'src'))
os.chdir(_NB)

OUT_DIR = Path('outputs/kriging')
OUT_DIR.mkdir(parents=True, exist_ok=True)

S3_BUCKET = 'thesis-data-ismaktam'
S3_ROOT   = 'kriging'

print(f'Project root: {_NB}')
print(f'S3:           s3://{S3_BUCKET}/{S3_ROOT}/')

Project root: /Users/etomengoi/Desktop/precip_interpolation_thesis
S3:           s3://thesis-data-ismaktam/kriging/


## 1. S3 helper + load variogram

In [2]:
s3 = boto3.client('s3')

def s3_download(s3_key: str, local_path: Path) -> bool:
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
        print(f'  ↓ s3://{S3_BUCKET}/{s3_key}')
        return True
    except Exception:
        return False

def fetch(s3_key: str, local_path: Path) -> bool:
    if local_path.exists():
        return True
    return s3_download(s3_key, local_path)

VGM_LOCAL = OUT_DIR / 'global_variogram.pkl'
VGM_S3    = f'{S3_ROOT}/global_variogram.pkl'
if not fetch(VGM_S3, VGM_LOCAL):
    raise FileNotFoundError(f'global_variogram.pkl not in S3 — run kriging_train.ipynb first')

with open(VGM_LOCAL, 'rb') as f:
    payload = pickle.load(f)
# Pickle from kriging_train is a wrapped payload {'variograms': {(t,vm): info}, 'empirical': ..., 'lag_centers_km': ...}
# Unwrap to the inner {(t, vm) -> info} dict so IND_KEY lookup works.
global_vgm = payload['variograms'] if isinstance(payload, dict) and 'variograms' in payload else payload
assert all(isinstance(k, tuple) and len(k) == 2 for k in global_vgm), \
    f'global_vgm has unexpected keys: {list(global_vgm)[:5]} — expected tuples of (transform, variogram_model)'

IND_KEY = ('indicator', 'spherical')
if IND_KEY not in global_vgm or global_vgm[IND_KEY] is None:
    raise ValueError(f'Indicator variogram missing from global_vgm. Keys: {list(global_vgm.keys())}')

ind_info = global_vgm[IND_KEY]
ind_params = ind_info['params_dict']
print(f'Global indicator variogram (spherical):')
print(f'  nugget = {ind_params["nugget"]:.4f}')
print(f'  psill  = {ind_params["psill"]:.4f}')
print(f'  sill   = {ind_params["nugget"] + ind_params["psill"]:.4f}')
print(f'  range  = {ind_params["range"]/1e3:.0f} km')

Global indicator variogram (spherical):
  nugget = 0.0589
  psill  = 0.0945
  sill   = 0.1534
  range  = 416 km


## 2. Load data + fit transforms

In [ ]:
from thesis.config import Config
from thesis.data.registry import DataRegistry
from thesis.transforms import (
    ProjectionTransform, IndicatorTransform, DetrendTransform,
)
from thesis.transforms.pipeline import TransformPipeline

cfg = Config()
registry = DataRegistry.from_config(cfg)

print('Loading ReKIS data (1966 train stations)...')
all_raw = registry.stations.load(cfg.date_start, cfg.date_end)
print(f'  Loaded {len(all_raw):,} rows, {all_raw["station_id"].nunique()} stations')

# Fit pipeline (projection + indicator + detrend)
proj = ProjectionTransform(target_crs=cfg.study_area.target_crs)
ind_t = IndicatorTransform(threshold_mm=cfg.wet_day_threshold_mm)
det  = DetrendTransform()

base_pipeline = TransformPipeline([proj, ind_t, det])
all_proc = base_pipeline.fit_transform(all_raw)
print(f'  Pipeline fitted. Shape: {all_proc.shape}')

Loading ReKIS data (1966 train stations)...


## 3. Build prediction grid + select days

In [ ]:
from thesis.datasets.protocols import PredictionGrid

grid = PredictionGrid.from_config(cfg, dem=registry.dem)
H, W = grid.shape
print(f'Grid: {grid.n_cells():,} cells, shape {grid.shape}')

# Find days with partial rainfall (30-70% wet stations) — interesting wet/dry boundaries
all_proc['date_str'] = all_proc['date'].astype(str)
daily_wet = all_proc.groupby('date_str')['rain_indicator'].mean()
interesting = daily_wet[(daily_wet > 0.25) & (daily_wet < 0.75)]
selected_dates = sorted(interesting.sample(min(3, len(interesting)), random_state=42).index.tolist())

print(f'\nSelected {len(selected_dates)} days for visualization:')
for d in selected_dates:
    frac = daily_wet[d]
    print(f'  {d}: wet fraction = {frac:.1%}')

## 4. Indicator kriging worker (uses global variogram)

In [ ]:
from thesis.models.kriging.ordinary import _build_pykrige_ok, _predict_chunked

def run_indicator_kriging(date_str: str):
    """Run Stage 1 indicator kriging for a single day using global variogram.
    
    Returns dict with p_rain, wet_grid, and station arrays.
    """
    day = all_proc[all_proc['date_str'] == date_str].copy()
    
    x_st = day['x_proj'].values
    y_st = day['y_proj'].values
    z_ind = day['rain_indicator'].values.astype(float)
    
    n_wet = int(z_ind.sum())
    n_total = len(z_ind)
    print(f'{date_str}: {n_total} stations, {n_wet} wet ({n_wet/n_total:.1%})')
    
    if n_wet == 0:
        print('  All-dry day → predict zero rain everywhere')
        p_rain = np.zeros((H, W))
        wet_grid = np.zeros((H, W), dtype=bool)
    elif n_wet == n_total:
        print('  All-wet day → predict rain everywhere')
        p_rain = np.ones((H, W))
        wet_grid = np.ones((H, W), dtype=bool)
    else:
        # Build indicator kriging with global variogram (Haylock 2008)
        ok = _build_pykrige_ok(x_st, y_st, z_ind, ind_info['model'], ind_params)
        
        # Predict P(rain) on full grid
        gx_flat = grid.coords_proj[:, 0]
        gy_flat = grid.coords_proj[:, 1]
        p_rain_flat, _ = _predict_chunked(ok, gx_flat, gy_flat)
        p_rain = np.clip(p_rain_flat, 0.0, 1.0).reshape(H, W)
        wet_grid = p_rain > cfg.kriging.indicator_probability_threshold
    
    print(f'  Grid wet cells: {wet_grid.sum():,} / {wet_grid.size:,} ({wet_grid.mean():.1%})')
    
    return {
        'p_rain': p_rain,
        'wet_grid': wet_grid,
        'stations': {'x': x_st, 'y': y_st, 'indicator': z_ind},
    }

print('Precomputed indicator kriging function.')

## 5. Run indicator kriging for selected days

In [ ]:
results = {}
for d in selected_dates:
    t0 = time.time()
    results[d] = run_indicator_kriging(d)
    print(f'  Elapsed: {time.time()-t0:.1f}s\n')

## 6. Global indicator variogram visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

nugget = ind_params['nugget']
psill  = ind_params['psill']
a      = ind_params['range']

# Fitted spherical curve
h = np.linspace(0, 500, 500)
h_m = h * 1e3
gamma_fit = np.where(
    h_m <= a,
    nugget + psill * (1.5 * h_m / a - 0.5 * (h_m / a) ** 3),
    nugget + psill,
)
gamma_fit[0] = 0

ax.plot(h, gamma_fit, color='#c0392b', lw=2.5, label='Spherical fit (global, Haylock 2008)')
ax.axhline(nugget + psill, color='gray', ls='--', lw=0.8, alpha=0.5, label=f'Sill = {nugget+psill:.4f}')
ax.axvline(a / 1e3, color='gray', ls=':', lw=0.8, alpha=0.5, label=f'Range = {a/1e3:.0f} km')

ax.set_xlabel('Distance $h$ (km)', fontsize=12)
ax.set_ylabel(r'Semi-variance $\hat{\gamma}(h)$', fontsize=12)
ax.set_title('Global Indicator Variogram (Spherical)\nPooled across all days (Stage 1 only)', fontsize=13)

info = (f'Nugget $c_0$ = {nugget:.4f}\nPartial sill $c_1$ = {psill:.4f}\n'
        f'Sill = {nugget+psill:.4f}\nRange $a$ = {a/1e3:.0f} km')
ax.text(0.95, 0.05, info, transform=ax.transAxes, fontsize=10,
        ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.9, edgecolor='gray'))

for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)
ax.grid(True, alpha=0.15)
ax.legend(loc='upper left', fontsize=10)
ax.set_xlim(0, 500)
ax.set_ylim(0, max(gamma_fit) * 1.1)

fig.tight_layout()
plt.show()
print(f'Indicator variogram parameters are the SAME for all days (global climatological fit).')

## 7. Zoomed-in visualization helper

In [ ]:
def find_boundary_cell(p_rain, threshold=0.4):
    """Find a grid cell near the wet/dry boundary (P ~ threshold)."""
    dist = np.abs(p_rain - threshold)
    margin = 80
    mask = np.ones_like(dist, dtype=bool)
    mask[:margin, :] = False
    mask[-margin:, :] = False
    mask[:, :margin] = False
    mask[:, -margin:] = False
    dist[~mask] = 999
    return np.unravel_index(np.argmin(dist), dist.shape)

def plot_indicator_day(date_str, result, zoom_radius_km=150):
    """3-panel figure: full P(rain) | zoomed stations+range | binary wet/dry."""
    r = result
    p_rain = r['p_rain']
    wet_grid = r['wet_grid']
    st = r['stations']
    vgm_range = ind_params['range']
    
    gx_2d = grid.coords_proj[:, 0].reshape(H, W)
    gy_2d = grid.coords_proj[:, 1].reshape(H, W)
    
    # Find boundary cell
    try:
        row_t, col_t = find_boundary_cell(p_rain)
    except:
        row_t, col_t = H // 2, W // 2
    
    target_x = gx_2d[row_t, col_t]
    target_y = gy_2d[row_t, col_t]
    
    zoom_r = zoom_radius_km * 1e3
    x_lo, x_hi = target_x - zoom_r, target_x + zoom_r
    y_lo, y_hi = target_y - zoom_r, target_y + zoom_r
    
    col_lo = max(0, np.searchsorted(gx_2d[0, :], x_lo) - 1)
    col_hi = min(W, np.searchsorted(gx_2d[0, :], x_hi) + 1)
    row_lo = max(0, np.searchsorted(gy_2d[:, 0], y_lo) - 1)
    row_hi = min(H, np.searchsorted(gy_2d[:, 0], y_hi) + 1)
    
    dist_to_target = np.sqrt((st['x'] - target_x)**2 + (st['y'] - target_y)**2)
    inside_range = dist_to_target <= vgm_range
    in_zoom = ((st['x'] >= x_lo) & (st['x'] <= x_hi) & (st['y'] >= y_lo) & (st['y'] <= y_hi))
    
    n_wet_st = int(st['indicator'].sum())
    n_total_st = len(st['indicator'])
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    fig.subplots_adjust(bottom=0.22)
    
    # --- Panel 1: Full domain P(rain) ---
    ax1 = axes[0]
    ext = [gx_2d[0,0]/1e3, gx_2d[0,-1]/1e3, gy_2d[0,0]/1e3, gy_2d[-1,0]/1e3]
    im1 = ax1.imshow(p_rain, cmap='RdYlBu_r', vmin=0, vmax=1, origin='lower', extent=ext)
    plt.colorbar(im1, ax=ax1, label='P(rain)', shrink=0.8)
    rect = plt.Rectangle((x_lo/1e3, y_lo/1e3), 2*zoom_r/1e3, 2*zoom_r/1e3,
                          lw=2.5, edgecolor='black', facecolor='none', ls='--')
    ax1.add_patch(rect)
    ax1.set_title(f'P(rain) — full domain\n{date_str}  ({n_wet_st}/{n_total_st} stations wet)', fontsize=12)
    ax1.set_xlabel('Easting (km)')
    ax1.set_ylabel('Northing (km)')
    
    # --- Panel 2: Zoomed P(rain) + stations + range circle ---
    ax2 = axes[1]
    zoom_ext = [x_lo/1e3, x_hi/1e3, y_lo/1e3, y_hi/1e3]
    p_zoom = p_rain[row_lo:row_hi, col_lo:col_hi]
    ax2.imshow(p_zoom, cmap='RdYlBu_r', vmin=0, vmax=1, origin='lower',
               extent=zoom_ext, alpha=0.7)
    
    circle = plt.Circle((target_x/1e3, target_y/1e3), vgm_range/1e3,
                         fill=False, edgecolor='#c0392b', lw=2.5, label='Variogram range')
    ax2.add_patch(circle)
    ax2.plot(target_x/1e3, target_y/1e3, marker='x', color='#c0392b',
             markersize=12, markeredgewidth=3, zorder=10, label='Target cell')
    
    for mask_sel, marker, alpha, label in [(in_zoom & inside_range, 'o', 0.9, 'in range'),
                                             (in_zoom & ~inside_range, 's', 0.4, 'outside range')]:
        if mask_sel.sum() == 0:
            continue
        wet_m = st['indicator'][mask_sel] == 1
        if wet_m.sum() > 0:
            ax2.scatter(st['x'][mask_sel][wet_m]/1e3, st['y'][mask_sel][wet_m]/1e3,
                       c='steelblue', marker=marker, s=50, alpha=alpha,
                       edgecolors='black', linewidths=0.5, zorder=8)
        if (~wet_m).sum() > 0:
            ax2.scatter(st['x'][mask_sel][~wet_m]/1e3, st['y'][mask_sel][~wet_m]/1e3,
                       c='#e8e8e8', marker=marker, s=50, alpha=alpha,
                       edgecolors='black', linewidths=0.5, zorder=8)
    
    n_in = inside_range.sum()
    n_in_wet = int((st['indicator'][inside_range] == 1).sum())
    ax2.set_title(f'Zoomed: stations in variogram range\n{n_in} stations used ({n_in_wet} wet, {n_in - n_in_wet} dry)', fontsize=12)
    ax2.set_xlim(zoom_ext[0], zoom_ext[1])
    ax2.set_ylim(zoom_ext[2], zoom_ext[3])
    ax2.set_xlabel('Easting (km)')
    ax2.set_aspect('equal')
    ax2.legend(fontsize=9, loc='upper left')
    
    # --- Panel 3: Binary wet/dry map (zoomed) ---
    ax3 = axes[2]
    wet_zoom = wet_grid[row_lo:row_hi, col_lo:col_hi].astype(float)
    cmap_wd = ListedColormap(['#f5f5f5', 'steelblue'])
    ax3.imshow(wet_zoom, cmap=cmap_wd, vmin=0, vmax=1, origin='lower', extent=zoom_ext)
    
    p_zoom_full = p_rain[row_lo:row_hi, col_lo:col_hi]
    xc = np.linspace(zoom_ext[0], zoom_ext[1], p_zoom_full.shape[1])
    yc = np.linspace(zoom_ext[2], zoom_ext[3], p_zoom_full.shape[0])
    ax3.contour(xc, yc, p_zoom_full, levels=[0.4], colors=['#c0392b'],
                linewidths=1.5, linestyles='--', label='P = 0.4 contour')
    
    wet_m = st['indicator'][in_zoom] == 1
    if wet_m.sum() > 0:
        ax3.scatter(st['x'][in_zoom][wet_m]/1e3, st['y'][in_zoom][wet_m]/1e3,
                   c='steelblue', marker='o', s=30, edgecolors='black',
                   linewidths=0.5, zorder=8, alpha=0.7)
    if (~wet_m).sum() > 0:
        ax3.scatter(st['x'][in_zoom][~wet_m]/1e3, st['y'][in_zoom][~wet_m]/1e3,
                   c='white', marker='o', s=30, edgecolors='black',
                   linewidths=0.5, zorder=8, alpha=0.7)
    
    ax3.set_title('Binary wet/dry classification\n(P > 0.4 threshold)', fontsize=12)
    ax3.set_xlim(zoom_ext[0], zoom_ext[1])
    ax3.set_ylim(zoom_ext[2], zoom_ext[3])
    ax3.set_xlabel('Easting (km)')
    ax3.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

print('Visualization function defined.')

## 8. Generate plots for all selected days

In [ ]:
for d in selected_dates:
    print(f"\n{'='*70}")
    print(f"  {d}")
    print(f"{'='*70}")
    plot_indicator_day(d, results[d], zoom_radius_km=75)

## 9. Summary statistics

In [ ]:
print('\n=== INDICATOR KRIGING SUMMARY ===')
print(f'\nGlobal indicator variogram (spherical, Haylock 2008 §3.3):')
print(f'  Nugget  = {ind_params["nugget"]:.4f}')
print(f'  Psill   = {ind_params["psill"]:.4f}')
print(f'  Sill    = {ind_params["nugget"] + ind_params["psill"]:.4f}')
print(f'  Range   = {ind_params["range"]/1e3:.0f} km')
print(f'\nThreshold for wet/dry: P(rain) > {cfg.kriging.indicator_probability_threshold}  (Hofstra 2008)')

rows = []
for d in selected_dates:
    r = results[d]
    st = r['stations']
    n_wet = int(st['indicator'].sum())
    n_total = len(st['indicator'])
    rows.append({
        'Date': d,
        'Wet stations': f"{n_wet}/{n_total}",
        'Station wet %': f"{n_wet/n_total:.1%}",
        'Grid wet cells': f"{r['wet_grid'].sum():,}",
        'Grid wet %': f"{r['wet_grid'].mean():.1%}",
    })

print('\n=== PER-DAY STATISTICS ===')
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

print('\n→ Stage 1 uses a SINGLE global variogram (same for all days) to predict P(rain).')
print('→ Stage 2 amount kriging varies by transform (none/log/normal_score) — see kriging_train.ipynb')